In [1]:
import os
import requests
from dotenv import load_dotenv

load_dotenv("../.env")
CLIENT_ID = os.environ["TWITCH_CLIENT_ID"]
CLIENT_SECRET = os.environ["TWITCH_CLIENT_SECRET"]

# 1. get an access token from Twitch
tok = requests.post("https://id.twitch.tv/oauth2/token", params={
    "client_id": CLIENT_ID,
    "client_secret": CLIENT_SECRET,
    "grant_type": "client_credentials",
}, timeout=30)
tok.raise_for_status()
token = tok.json()["access_token"]
print("Token received. Expires in", tok.json()["expires_in"] // 86400, "days")

# 2. test request to IGDB
headers = {"Client-ID": CLIENT_ID, "Authorization": f"Bearer {token}"}
body = """
fields name, first_release_date, aggregated_rating, aggregated_rating_count, platforms.name;
search "Celeste";
limit 3;
"""
r = requests.post("https://api.igdb.com/v4/games", headers=headers, data=body, timeout=30)
print(r.status_code)
print(r.json())

Token received. Expires in 57 days
200
[{'id': 26226, 'aggregated_rating': 91.3, 'aggregated_rating_count': 10, 'first_release_date': 1516838400, 'name': 'Celeste', 'platforms': [{'id': 170, 'name': 'Google Stadia'}, {'id': 48, 'name': 'PlayStation 4'}, {'id': 3, 'name': 'Linux'}, {'id': 6, 'name': 'PC (Microsoft Windows)'}, {'id': 14, 'name': 'Mac'}, {'id': 49, 'name': 'Xbox One'}, {'id': 130, 'name': 'Nintendo Switch'}]}, {'id': 76427, 'first_release_date': 1510531200, 'name': 'Celeste Witch', 'platforms': [{'id': 6, 'name': 'PC (Microsoft Windows)'}]}, {'id': 91484, 'first_release_date': 473299200, 'name': 'Marie Celeste', 'platforms': [{'id': 26, 'name': 'ZX Spectrum'}]}]


In [2]:
import pandas as pd

body = "fields id, name, generation, platform_family.name; limit 500;"
r = requests.post("https://api.igdb.com/v4/platforms", headers=headers, data=body, timeout=30)
plats = pd.DataFrame(r.json())

plats["family"] = plats["platform_family"].apply(lambda x: x["name"] if isinstance(x, dict) else None)
console_plats = plats[plats["family"].isin(["Nintendo", "PlayStation", "Xbox"])]
console_plats.sort_values(["family", "generation"])[["id", "name", "family", "generation"]]

,id,name,family,generation
97,307,Game & Watch,Nintendo,2.0
39,51,Family Computer Disk System,Nintendo,3.0
76,18,Nintendo Entertainment System,Nintendo,3.0
145,99,Family Computer,Nintendo,3.0
6,306,Satellaview,Nintendo,4.0
92,131,Super NES CD-ROM System,Nintendo,4.0
112,33,Game Boy,Nintendo,4.0
193,19,Super Nintendo Entertainment System,Nintendo,4.0
195,58,Super Famicom,Nintendo,4.0
26,87,Virtual Boy,Nintendo,5.0


In [3]:
plats[["id", "name", "family", "generation"]].to_csv("../data/processed/igdb_platforms.csv", index=False)